# Timetabling December Examinations


Timetabling is the process of best fitting a set of classes or labs or exams into a set of rooms, under a number of constraints. Usually the rooms have a limiting capacity, the classes have a fixed number of students, the classes or exams or labs take a fixed length of time, and clearly a room cannot exist in two places at once, and a student cannot be in two places at once.

The problem is what we call *NP-hard*. That doesn't mean that the problem is 'hard' or 'complex' in the 'rocket science' sense. But it does mean that the number of options increases drastically as the number of classes or rooms increases, and so if we searched through all the possible options exhaustively to find the best solution, it could only be done for a small number of classes. There are so many options for a large number of classes that finding the best solution is like finding a needle in a field of haystacks. So in practice we don't look for the best solution by an exhaustive process. Instead we find an algorithm / procedure to obtain a decent solution, and leave it at that. 

Your task is to take information on a list of exams, and on a list of exam halls (both in .csv files)

* Dec2012Exams.csv
* MainExamRooms.csv

and produce a spreadsheet which contains the exam dates and times, and checks to see if all December exams can fit in the specified exam period. You may use any algorithm that you think appropriate - including just exhaustive search. 

#### Constraints and Simplifications

The most critical simplification of the problem is to assume that there are no joint degrees, and so any `ENGxxxx` class *can* be timetabled at the same time as a `BIOLxxxx` class. We also simplify by saying that no `ENG1xxx` class can be scheduled at the same time as another `ENG1xxx` class, which ensures that a student in 1st year Engineering cannot have two exams on top of one another. *(You can imagine the additional complexity added by joint degrees, and visting students taking a mix of exams from different years.)*
You can decide whether you are going to allow multiple exams in the same large room at the same time, and whether you will allow those exams to start and stop at different times in the same hall (annoying students in the process).
You can also decide whether you want to allow exam hall splits - where a class is spread over two exam halls.

#### Other Useful information
* Exams start at 09:30 or 13:00 in Glasgow, and multiple start times would annoy and confuse the janitorial staff and foul up organisation for those with additional time due to special needs
* Exams stretch from Wednesday 11th - Friday 20th December inclusive, and Saturday exams are possible.  
* Rules are that no student can have more than three exams in two days. 
* If you want more information about practical exam timetabling, then we've included a conference paper in the zip file - but don't do something even close to that complex


If you really want additional complications, invent a spreadsheet of additional constraints - invent a few exams that are contemporaneous - i.e. Exams such as Analogue Electronics 1 and EE1X have some identical material and therefore have to be run at the same time, even although they are technically different exams. Or you can invent a list of special exams that have to be run in the morning to allow them to synchronise with students takign  them abroad.

In [ ]:
#import tarrifs
import csv
import numpy as np

'''
this is an nphardtype "ETP" which deals with timetable generation in a university setting, we are using data from glasgow university
'''
#what follows is a proof of concept prototype to allow me to decide to do this or do the other lab 8 ones

#setup exam class type
class Exam:
    """Represents a single University exam from Dec2012Exams.csv."""
    def __init__(self, code, name, students, length):
        self.code = code.strip()
        self.name = name.strip()
        self.students = int(students)
        self.length = float(length)
        self.cohort = self._extract_cohort(self.code)
        
        # Scheduling results
        self.assigned_session = None
        self.assigned_rooms = []

    def _extract_cohort(self, code):
        #Extracts the Subject + Year cohort (e.g. 'ENG1010_1' -> 'ENG1') joint we are told do not exsists
        year = ""
        for char in code:
            if char.isalpha() and year == "":
                dept += char
            elif char.isdigit():
                year = char
                break
        return dept + year


class Room:
    #Represents an exam hall from MainExamRooms.csv.
    def __init__(self, name, capacity):
        self.name = name.strip()
        self.capacity = int(capacity)


class Session: #Represents a single exam session (e.g. 'Wed 11 Dec - 09:30').
    #track remaining room seats room exam lengths and sched chorts
    def __init__(self, session_id, day_index, date_str, time_str, rooms):
        self.session_id = session_id
        self.day_index = day_index      # 0 to 8 (for the 9 exam days)
        self.date_str = date_str
        self.time_str = time_str
        
        # Track remaining capacity in each room for this session
        self.room_remaining = {r.name: r.capacity for r in rooms}
        # Track the exam duration currently running in each room (None if empty)
        self.room_duration = {r.name: None for r in rooms}
        # Track which cohorts already have an exam in this session
        self.cohorts_in_session = set()
        self.scheduled_exams = []

    def allocate_rooms(self, exam, same_length_only=True, allow_splits=True):
        """
        okay this one is difficult, we are trying to find a room for the exam in this session
        attempts to find room(s) for 'exam' in this session are given by this policy:
        1. Try to fit the entire exam into a single room (Best-Fit: smallest room
        that is large enough, so we save Bute/Hunter Hall for huge classes).
        2. If sharing a room with an existing exam, require the SAME exam length
        so students leaving early do not disturb others.
        3. If no single room is big enough and allow_splits=True, split the exam
        across the 2 largest available compatible rooms.
        Returns a list of (room_name, seats_used) if successful, or None if not.
        """
        compatible_rooms = []
        for r_name, rem_cap in self.room_remaining.items():
            if rem_cap <= 0:
                continue
            cur_len = self.room_duration[r_name]
            # Room is either completely empty, or already hosting an exam of the exact same length
            if (cur_len is None) or (not same_length_only) or (cur_len == exam.length):
                compatible_rooms.append((r_name, rem_cap))

        # 1. Try single-room Best-Fit (sort compatible rooms from smallest to largest capacity)
        single_candidates = [r for r in compatible_rooms if r[1] >= exam.students]
        if len(single_candidates) > 0:
            single_candidates.sort(key=lambda x: x[1])
            chosen_room = single_candidates[0][0]
            return [(chosen_room, exam.students)]

        # 2. If single room fails, try splitting across 2 rooms (largest available first)
        if allow_splits and len(compatible_rooms) >= 2:
            compatible_rooms.sort(key=lambda x: x[1], reverse=True)
            r1_name, r1_cap = compatible_rooms[0]
            r2_name, r2_cap = compatible_rooms[1]
            if (r1_cap + r2_cap) >= exam.students:
                seats_in_r1 = r1_cap
                seats_in_r2 = exam.students - seats_in_r1
                return [(r1_name, seats_in_r1), (r2_name, seats_in_r2)]

        return None

    def commit_exam(self, exam, room_allocations):
        #comt the exam to this session and updates room capacities
        for r_name, seats_used in room_allocations:
            self.room_remaining[r_name] -= seats_used
            self.room_duration[r_name] = exam.length
        self.cohorts_in_session.add(exam.cohort)
        self.scheduled_exams.append(exam)
        exam.assigned_session = self
        exam.assigned_rooms = room_allocations


# eat data

def load_exams(filename):
    exams = []
    # utf-8-sig automatically strips the hidden BOM character at the start of the CSV, this was very annoying to discover
    with open(filename, mode='r', encoding='utf-8-sig') as f:
        reader = csv.DictReader(f)
        for row in reader:
            exams.append(Exam(
                row['Exam Number'],
                row['Full Name'],
                row['Students'],
                row['Length (h)']
            ))
    return exams

#func to load rooms from csv file
def load_rooms(filename):
    rooms = []
    with open(filename, mode='r', encoding='utf-8-sig') as f:
        reader = csv.DictReader(f)
        for row in reader:
            rooms.append(Room(row['Room'], row['Teaching Capacity']))
    return rooms


#TIMETABLE SCHEDULER (Sequential Construction Heuristic)


def create_december_sessions(rooms):
    """
    Creates the 18 exam sessions from Wed 11th Dec to Fri 20th Dec
    (including Saturday 14th Dec, excluding Sunday 15th Dec).
    """
    dates = [
        "Wed 11 Dec", "Thu 12 Dec", "Fri 13 Dec", "Sat 14 Dec",
        # Sunday 15 Dec excluded
        "Mon 16 Dec", "Tue 17 Dec", "Wed 18 Dec", "Thu 19 Dec", "Fri 20 Dec"
    ]
    times = ["09:30", "13:00"]
    
    sessions = []
    session_id = 0
    for day_idx, date_str in enumerate(dates):
        for time_str in times:
            sessions.append(Session(session_id, day_idx, date_str, time_str, rooms))
            session_id += 1
    return sessions


def violates_three_in_two_days(cohort, candidate_session, all_sessions):
    """
    we should go back to joint exams btw
    anyway 
    this is a hard constraint that says no more than 3 exams in any 2 consecutive days for the same cohort
    so we check the candidate session and the sessions on the day before and after to see if it would violate this constraint
    """
    d = candidate_session.day_index
    # Check window [d - 1, d] and window [d, d + 1]
    for start_day in [d - 1, d]:
        end_day = start_day + 1
        exams_in_window = 0
        for s in all_sessions:
            if start_day <= s.day_index <= end_day:
                if cohort in s.cohorts_in_session:
                    exams_in_window += 1
        # Adding candidate_session would make it exams_in_window + 1
        if exams_in_window + 1 > 3:
            return True
    return False

#now we create a function to schedule exams into sessions and rooms
def schedule_exams(exams, rooms):
    sessions = create_december_sessions(rooms)

    # Count how many exams belong to each cohort so we schedule the most
    # constrained cohorts (like ENG3, ENG2, ENG4) FIRST (Welsh-Powell seems to be what is used in all the papers i read, which will be in the bib)
    cohort_counts = {}
    for e in exams:
        cohort_counts[e.cohort] = cohort_counts.get(e.cohort, 0) + 1

    # Sort exams by: 1) Cohort size (descending), 2) Student count (descending)
    sorted_exams = sorted(
        exams,
        key=lambda e: (cohort_counts[e.cohort], e.students),
        reverse=True
    )

    unscheduled_strict = []

    # strict schd loop
    for exam in sorted_exams:
        scheduled = False
        for session in sessions:
            # Hard Constraint 1: No cohort clash in the same session (cite here)
            if exam.cohort in session.cohorts_in_session:
                continue
            # Hard Constraint 2: Max 3 exams in any 2 consecutive days (cite here)
            if violates_three_in_two_days(exam.cohort, session, sessions):
                continue
            # Hard Constraint 3: Room capacity & duration match (cite here)
            room_alloc = session.allocate_rooms(exam, same_length_only=True, allow_splits=True)
            if room_alloc is not None:
                session.commit_exam(exam, room_alloc)
                scheduled = True
                break

        if not scheduled:# if we could not schedule the exam in any session, we add it to the unscheduled list
            unscheduled_strict.append(exam)

    return sessions, unscheduled_strict, cohort_counts


#what follows is the primary entry point for the program, it loads the exams and rooms from csv files, runs the scheduler, and outputs a report and a csv file with the results

if __name__ == "__main__": #we take in the csv files and run the scheduler
    exams = load_exams("Dec2012Exams.csv")
    rooms = load_rooms("MainExamRooms.csv")

    total_seats_per_session = sum(r.capacity for r in rooms)
    max_single_room = max(r.capacity for r in rooms) #for the report, we need to know the largest single room capacity

    print("=" * 70)
    print(" GLASGOW DECEMBER EXAM TIMETABLING - PRIMARY REPORT")
    print("=" * 70)
    print("Total Exams to Schedule     : {}".format(len(exams)))
    print("Total Exam Halls Available  : {} (Max single room: {} seats)".format(len(rooms), max_single_room))
    print("Total Seats per Session     : {} seats".format(total_seats_per_session))

    # Check for oversized exams (> max single room)
    oversized = [e for e in exams if e.students > max_single_room]
    print("\n[!] Exams exceeding largest single hall ({} seats) -> Require Hall Split:".format(max_single_room))
    for e in oversized:
        print("    - {:<14} ({:<30}) : {} students".format(e.code, e.name, e.students))

    # Run scheduler
    sessions, unscheduled, cohort_counts = schedule_exams(exams, rooms)

    print("\n" + "=" * 70) #create table
    print(" SCHEDULING RESULTS (18 Sessions: Wed 11 Dec - Fri 20 Dec)")
    print("=" * 70)
    print("Successfully Scheduled (Strict Rules) : {} / {} exams".format(
        len(exams) - len(unscheduled), len(exams) #logic is that we take the total number of exams and subtract the unscheduled ones to get the scheduled ones
    ))
    print("Unscheduled Overflow Exams            : {} exams".format(len(unscheduled)))

    if len(unscheduled) > 0:
        print("\nWhy did {} exams overflow? Cohort Bottleneck Analysis:".format(len(unscheduled)))
        print("(Note: In 9 days / 18 sessions, 'Max 3 exams in 2 days' allows at most 14 exams per cohort!)")
        overflow_cohorts = {}
        for e in unscheduled:
            overflow_cohorts[e.cohort] = overflow_cohorts.get(e.cohort, 0) + 1
        for cohort, count in sorted(overflow_cohorts.items(), key=lambda x: x[1], reverse=True):
            print("    - Cohort {:<8} : {} total exams ({} scheduled, {} overflowed)".format(
                cohort, cohort_counts[cohort], cohort_counts[cohort] - count, count
            ))

    # Save output spreadsheet
    output_filename = "December_Exam_Timetable.csv"
    with open(output_filename, mode='w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow([
            "Exam Code", "Full Name", "Cohort", "Students", "Length (h)",
            "Status", "Date", "Time", "Assigned Room(s)"
        ])
        for s in sessions:
            for e in s.scheduled_exams:
                room_str = " + ".join(["{} ({})".format(r, c) for r, c in e.assigned_rooms])
                writer.writerow([
                    e.code, e.name, e.cohort, e.students, e.length,
                    "SCHEDULED", s.date_str, s.time_str, room_str
                ])
        for e in unscheduled:
            writer.writerow([
                e.code, e.name, e.cohort, e.students, e.length,
                "OVERFLOW (Cohort Limit Exceeded)", "UNASSIGNED", "UNASSIGNED", "NONE"
            ]) #this is for the ones that got away

    print("\n[+] Full timetable saved to '{}'".format(output_filename)) #genieuely forgot the newline command for like 30 seconds wirintg this btw, i am cooked